In [0]:
%sql
USE CATALOG retail_dwh;
USE SCHEMA gold;

 Highest revenue month

In [0]:
%sql
SELECT DATE_FORMAT(TxnDate,'yyyy-MM') AS Month,
       ROUND(SUM(Amount),2) AS Total_Revenue
FROM FactSales
GROUP BY Month
ORDER BY Total_Revenue DESC
LIMIT 1;

Monthly Revenue

In [0]:
%sql
SELECT DATE_FORMAT(TxnDate,'yyyy-MM') AS Month,
       COUNT(TransactionID) AS Total_Transactions,
       ROUND(SUM(Amount),2) AS Revenue
FROM FactSales
GROUP BY Month
ORDER BY Month;

Top customers by spending

In [0]:
%sql
SELECT c.CustomerID,
       c.CustomerName,
       c.City,
       ROUND(SUM(f.Amount),2) AS Total_Spent
FROM FactSales f
JOIN DimCustomer c
ON f.CustomerSK = c.CustomerSK
WHERE c.IsActive = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
ORDER BY Total_Spent DESC
LIMIT 10;

Products that generate top revenue

In [0]:
%sql
SELECT p.ProductID,
       p.ProductName,
       p.Category,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimProduct p
ON f.ProductSK = p.ProductSK
GROUP BY p.ProductID, p.ProductName, p.Category
ORDER BY Revenue DESC
LIMIT 10;

Best selling product categories

In [0]:
%sql
SELECT p.Category,
       SUM(f.Quantity) AS Units_Sold,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimProduct p
ON f.ProductSK = p.ProductSK
GROUP BY p.Category
ORDER BY Revenue DESC;

Revenue by region

In [0]:
%sql
SELECT s.Region,
       COUNT(f.TransactionID) AS Transactions,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimStore s
ON f.StoreSK = s.StoreSK
GROUP BY s.Region
ORDER BY Revenue DESC;

In [0]:
%sql
Average ordervalue

In [0]:
%sql
SELECT ROUND(AVG(Amount),2) AS Avg_Order_Value
FROM FactSales;

Products never sold

In [0]:
%sql
SELECT p.ProductID,
       p.ProductName,
       p.Category
FROM DimProduct p
LEFT JOIN FactSales f
ON p.ProductSK = f.ProductSK
WHERE f.ProductSK IS NULL;

Customer history

In [0]:
%sql
SELECT CustomerID,
       CustomerName,
       City,
       Address,
       StartDate,
       EndDate,
       CASE
            WHEN IsActive = 1 THEN 'Current'
            ELSE 'Historical'
       END AS Record_Status
FROM DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM DimCustomer
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;